# FP&A Variance Report Generator — Stage 2: Data Validation

Goal: load raw input, catch bad data BEFORE calculating variance.
6 checks. Each check = one cell, one output, one reason logged.

In [1]:
import pandas as pd

sheets = pd.read_excel("sample_input.xlsx", sheet_name=None)
actuals = sheets["Actuals"]
budget = sheets["Budget"]
prior = sheets["Prior_Month"]
department = sheets["Department"]
control = sheets["Control_Totals"]

actuals.shape, actuals.columns.tolist()

((110, 5),
 ['month', 'cost_center', 'department', 'account_category', 'actual_amount'])

## Exceptions log

One table. Every issue any check finds gets a row here: which sheet, which row, what's wrong.
This table = your "exceptions requiring human review" deliverable.

In [2]:
exceptions = []

def flag(sheet, row_idx, issue, detail):
    exceptions.append({"sheet": sheet, "row": row_idx, "issue": issue, "detail": detail})

## Check 1 — missing cost center

In [3]:
missing_cc = actuals[actuals["cost_center"].isna() | (actuals["cost_center"] == "")]
for idx in missing_cc.index:
    flag("Actuals", idx, "missing_cost_center", "cost_center blank")

missing_cc

,month,cost_center,department,account_category,actual_amount
36,2026-06-01,NaN,Manufacturing,Revenue,4700079.07
37,2026-06-01,NaN,Manufacturing,Raw Material Cost,1618237.00


## Check 2 — missing account category

In [4]:
missing_cat = actuals[actuals["account_category"].isna() | (actuals["account_category"] == "")]
for idx in missing_cat.index:
    flag("Actuals", idx, "missing_account_category", "account_category blank")

missing_cat

,month,cost_center,department,account_category,actual_amount
72,2026-08-01,CC-100,Manufacturing,NaN,4837317.64
73,2026-08-01,CC-100,Manufacturing,NaN,1787369.81


## Check 3 — duplicate records
Same month + cost_center + account_category appearing twice = duplicate entry.

In [5]:
dup_mask = actuals.duplicated(subset=["month","cost_center","account_category"], keep="first")
dupes = actuals[dup_mask]
for idx in dupes.index:
    flag("Actuals", idx, "duplicate_record", "same month/cost_center/category seen earlier")

dupes

,month,cost_center,department,account_category,actual_amount
73,2026-08-01,CC-100,Manufacturing,NaN,1787369.81
108,2026-04-01,CC-100,Manufacturing,Admin Overheads,253841.32
109,2026-05-01,CC-100,Manufacturing,Employee Cost,958343.38


## Check 4 — invalid dates
`pd.to_datetime(..., errors="coerce")` turns anything unparseable into NaT (Not a Time).

In [6]:
parsed_dates = pd.to_datetime(actuals["month"], errors="coerce")
bad_dates = actuals[parsed_dates.isna()]
for idx in bad_dates.index:
    flag("Actuals", idx, "invalid_date", f"unparseable month value: {actuals.loc[idx,'month']}")

bad_dates

,month,cost_center,department,account_category,actual_amount
54,2026-13-45,CC-100,Manufacturing,Revenue,4870979.02


## Check 5 — unexpected negative values
Revenue should never be negative. (Cost categories CAN legitimately be negative — e.g. a credit/refund — so we only flag Revenue here.)

In [7]:
neg_revenue = actuals[(actuals["account_category"] == "Revenue") & (actuals["actual_amount"] < 0)]
for idx in neg_revenue.index:
    flag("Actuals", idx, "unexpected_negative", f"Revenue negative: {actuals.loc[idx,'actual_amount']}")

neg_revenue

,month,cost_center,department,account_category,actual_amount
90,2026-09-01,CC-100,Manufacturing,Revenue,-5148111.05


## Check 6 — control total mismatch
Sum of Actuals per month should equal Control_Totals. Mismatch = something missing/duplicated upstream, or a genuine data entry error.

In [8]:
# exclude rows already flagged as bad data (blank cc/category, bad dates) from the sum,
# so one bad row doesn't mask or fake a control-total match
clean_actuals = actuals.drop(index=set(missing_cc.index) | set(missing_cat.index) | set(bad_dates.index) | set(dupes.index))

monthly_actual = clean_actuals.groupby("month")["actual_amount"].sum().reset_index()
monthly_actual.columns = ["month", "sum_of_actuals"]

check = control.merge(monthly_actual, on="month", how="left")
check["diff"] = (check["control_total_actual"] - check["sum_of_actuals"]).round(2)
mismatches = check[check["diff"].abs() > 1]  # tolerance ₹1 for rounding

for _, r in mismatches.iterrows():
    flag("Control_Totals", r["month"], "control_total_mismatch", f"control={r['control_total_actual']}, sum_actuals={r['sum_of_actuals']}, diff={r['diff']}")

check

,month,control_total_actual,sum_of_actuals,diff
0,2026-04-01,18390817.12,18390817.12,0.00
1,2026-05-01,18779869.18,18779869.18,0.00
2,2026-06-01,18034424.48,11716108.41,6318316.07
3,2026-07-01,18859977.77,13988998.75,4870979.02
4,2026-08-01,18869587.37,12194899.92,6674687.45
5,2026-09-01,19445508.33,9149286.23,10296222.10


## All exceptions — the review log

In [9]:
exceptions_df = pd.DataFrame(exceptions)
print(f"Total exceptions found: {len(exceptions_df)}")
exceptions_df

Total exceptions found: 13


,sheet,row,issue,detail
0,Actuals,36,missing_cost_center,cost_center blank
1,Actuals,37,missing_cost_center,cost_center blank
2,Actuals,72,missing_account_category,account_category blank
3,Actuals,73,missing_account_category,account_category blank
4,Actuals,73,duplicate_record,same month/cost_center/category seen earlier
5,Actuals,108,duplicate_record,same month/cost_center/category seen earlier
6,Actuals,109,duplicate_record,same month/cost_center/category seen earlier
7,Actuals,54,invalid_date,unparseable month value: 2026-13-45
8,Actuals,90,unexpected_negative,Revenue negative: -5148111.05
9,Control_Totals,2026-06-01,control_total_mismatch,"control=18034424.48, sum_actuals=11716108.41, ..."


## Clean dataset for Stage 3

`clean_actuals` = actuals with the row-level bad data removed (blank cc/category, bad dates, duplicates).
Control-total mismatches are logged but NOT auto-corrected — that needs a human, which is exactly the point.
This clean set feeds the variance calculations next.

In [10]:
print("Original rows:", len(actuals))
print("Clean rows after validation:", len(clean_actuals))
print("Exceptions logged:", len(exceptions_df))

clean_actuals.head()

Original rows: 110
Clean rows after validation: 103
Exceptions logged: 13


,month,cost_center,department,account_category,actual_amount
0,2026-04-01,CC-100,Manufacturing,Revenue,4723064.29
1,2026-04-01,CC-100,Manufacturing,Raw Material Cost,1712428.40
2,2026-04-01,CC-100,Manufacturing,Employee Cost,923319.50
3,2026-04-01,CC-100,Manufacturing,Marketing Expense,406668.13
4,2026-04-01,CC-100,Manufacturing,Logistics Cost,315239.26


---
# Stage 3 — Variance Calculations

Input: `clean_actuals` (validated rows only) from Stage 2.
Output: variance table with abs/% variance, fav/unfav flag, MoM change, top drivers.

## Join Actuals + Budget + Prior_Month
Match on month + cost_center + account_category. Left join from clean_actuals — budget/prior rows with no matching actual just won't appear (expected, not an error).

In [11]:
keys = ["month", "cost_center", "account_category"]

variance = clean_actuals.merge(budget[keys + ["budget_amount"]], on=keys, how="left")
variance = variance.merge(prior[keys + ["prior_month_amount"]], on=keys, how="left")

variance.head()

,month,cost_center,department,account_category,actual_amount,budget_amount,prior_month_amount
0,2026-04-01,CC-100,Manufacturing,Revenue,4723064.29,4172850.49,4356181.73
1,2026-04-01,CC-100,Manufacturing,Raw Material Cost,1712428.40,1763187.75,1556073.21
2,2026-04-01,CC-100,Manufacturing,Employee Cost,923319.50,840329.48,877647.99
3,2026-04-01,CC-100,Manufacturing,Marketing Expense,406668.13,425075.30,388162.35
4,2026-04-01,CC-100,Manufacturing,Logistics Cost,315239.26,345070.28,283920.20


## Variance metrics
- `abs_variance` = actual − budget
- `pct_variance` = abs_variance / budget (guard budget=0 → avoid divide-by-zero)
- `mom_change` = actual − prior month, `mom_pct_change` = that / prior month

In [12]:
variance["abs_variance"] = (variance["actual_amount"] - variance["budget_amount"]).round(2)
variance["pct_variance"] = (variance["abs_variance"] / variance["budget_amount"].replace(0, pd.NA)).round(4)

variance["mom_change"] = (variance["actual_amount"] - variance["prior_month_amount"]).round(2)
variance["mom_pct_change"] = (variance["mom_change"] / variance["prior_month_amount"].replace(0, pd.NA)).round(4)

variance[["month","cost_center","account_category","actual_amount","budget_amount","abs_variance","pct_variance"]].head()

,month,cost_center,account_category,actual_amount,budget_amount,abs_variance,pct_variance
0,2026-04-01,CC-100,Revenue,4723064.29,4172850.49,550213.80,0.1319
1,2026-04-01,CC-100,Raw Material Cost,1712428.40,1763187.75,-50759.35,-0.0288
2,2026-04-01,CC-100,Employee Cost,923319.50,840329.48,82990.02,0.0988
3,2026-04-01,CC-100,Marketing Expense,406668.13,425075.30,-18407.17,-0.0433
4,2026-04-01,CC-100,Logistics Cost,315239.26,345070.28,-29831.02,-0.0864


## Favorable / Unfavorable flag — business rule
Direction of "good" depends on category type:
- **Revenue** (income): actual > budget = **Favorable** (more income than planned)
- **Cost categories**: actual < budget = **Favorable** (spent less than planned)

This mapping is the one business judgment call in the whole pipeline — document it, since it's exactly what an interviewer will ask about.

In [13]:
income_categories = {"Revenue"}

def fav_flag(row):
    if row["account_category"] in income_categories:
        return "Favorable" if row["abs_variance"] > 0 else "Unfavorable"
    else:
        return "Favorable" if row["abs_variance"] < 0 else "Unfavorable"

variance["variance_flag"] = variance.apply(fav_flag, axis=1)
variance["variance_flag"].value_counts()

variance_flag
Unfavorable    52
Favorable      51
Name: count, dtype: int64

## Top 5 adverse and favorable drivers
Ranked by **magnitude** of abs_variance within each flag — the biggest swings, not just any unfavorable/favorable row.

In [14]:
top5_adverse = (variance[variance["variance_flag"]=="Unfavorable"]
                 .assign(magnitude=lambda d: d["abs_variance"].abs())
                 .sort_values("magnitude", ascending=False)
                 .head(5)[["month","cost_center","account_category","actual_amount","budget_amount","abs_variance","pct_variance"]])

top5_favorable = (variance[variance["variance_flag"]=="Favorable"]
                   .assign(magnitude=lambda d: d["abs_variance"].abs())
                   .sort_values("magnitude", ascending=False)
                   .head(5)[["month","cost_center","account_category","actual_amount","budget_amount","abs_variance","pct_variance"]])

print("TOP 5 ADVERSE")
display(top5_adverse)
print("TOP 5 FAVORABLE")
display(top5_favorable)

TOP 5 ADVERSE


,month,cost_center,account_category,actual_amount,budget_amount,abs_variance,pct_variance
85,2026-09-01,CC-100,Revenue,-5148111.05,4869616.64,-10017727.69,-2.0572
46,2026-06-01,CC-300,Revenue,2076413.02,2296349.90,-219936.88,-0.0958
79,2026-08-01,CC-300,Revenue,2127141.99,2329471.39,-202329.40,-0.0869
19,2026-05-01,CC-100,Raw Material Cost,1785832.17,1624606.49,161225.68,0.0992
80,2026-08-01,CC-300,Raw Material Cost,832567.95,724131.97,108435.98,0.1497


TOP 5 FAVORABLE


,month,cost_center,account_category,actual_amount,budget_amount,abs_variance,pct_variance
0,2026-04-01,CC-100,Revenue,4723064.29,4172850.49,550213.80,0.1319
6,2026-04-01,CC-200,Revenue,3605797.75,3138893.43,466904.32,0.1487
18,2026-05-01,CC-100,Revenue,4971684.11,4620983.60,350700.51,0.0759
91,2026-09-01,CC-200,Revenue,3555595.23,3363682.95,191912.28,0.0571
25,2026-05-01,CC-200,Raw Material Cost,1122797.42,1283928.21,-161130.79,-0.1255


## Full variance table — Stage 3 output

In [15]:
variance_final = variance[["month","cost_center","department","account_category",
                            "actual_amount","budget_amount","abs_variance","pct_variance",
                            "variance_flag","prior_month_amount","mom_change","mom_pct_change"]]

print(variance_final.shape)
variance_final.head(10)

(103, 12)


,month,cost_center,department,account_category,actual_amount,budget_amount,abs_variance,pct_variance,variance_flag,prior_month_amount,mom_change,mom_pct_change
0,2026-04-01,CC-100,Manufacturing,Revenue,4723064.29,4172850.49,550213.80,0.1319,Favorable,4356181.73,366882.56,0.0842
1,2026-04-01,CC-100,Manufacturing,Raw Material Cost,1712428.40,1763187.75,-50759.35,-0.0288,Favorable,1556073.21,156355.19,0.1005
2,2026-04-01,CC-100,Manufacturing,Employee Cost,923319.50,840329.48,82990.02,0.0988,Unfavorable,877647.99,45671.51,0.0520
3,2026-04-01,CC-100,Manufacturing,Marketing Expense,406668.13,425075.30,-18407.17,-0.0433,Favorable,388162.35,18505.78,0.0477
4,2026-04-01,CC-100,Manufacturing,Logistics Cost,315239.26,345070.28,-29831.02,-0.0864,Favorable,283920.20,31319.06,0.1103
5,2026-04-01,CC-100,Manufacturing,Admin Overheads,253841.32,245683.46,8157.86,0.0332,Unfavorable,232403.90,21437.42,0.0922
6,2026-04-01,CC-200,Sales & Marketing,Revenue,3605797.75,3138893.43,466904.32,0.1487,Favorable,3280091.94,325705.81,0.0993
7,2026-04-01,CC-200,Sales & Marketing,Raw Material Cost,1266718.92,1257382.45,9336.47,0.0074,Unfavorable,1232483.53,34235.39,0.0278
8,2026-04-01,CC-200,Sales & Marketing,Employee Cost,683941.62,722590.09,-38648.47,-0.0535,Favorable,653303.81,30637.81,0.0469
9,2026-04-01,CC-200,Sales & Marketing,Marketing Expense,336865.52,337862.32,-996.80,-0.0030,Favorable,322627.97,14237.55,0.0441


---
# Stage 4 — Excel Output + Management Summary

Two deliverables from here:
1. `validated_output.xlsx` — multi-sheet workbook, the "clean data product"
2. `management_summary.pdf` — one-page narrative, text generated FROM the numbers (not hand-written), so it stays correct if the input data changes.

## Monthly summary (needed for both outputs)

In [16]:
monthly_summary = variance_final.groupby("month").agg(
    total_actual=("actual_amount","sum"),
    total_budget=("budget_amount","sum"),
    total_variance=("abs_variance","sum"),
    unfavorable_count=("variance_flag", lambda s: (s=="Unfavorable").sum()),
    favorable_count=("variance_flag", lambda s: (s=="Favorable").sum()),
).round(2).reset_index()

monthly_summary

,month,total_actual,total_budget,total_variance,unfavorable_count,favorable_count
0,2026-04-01,18390817.12,17355996.88,1034820.24,8,10
1,2026-05-01,18779869.18,18316984.50,462884.68,8,10
2,2026-06-01,11716108.41,12148831.73,-432723.32,6,10
3,2026-07-01,13988998.75,14065003.01,-76004.26,9,8
4,2026-08-01,12194899.92,12285539.87,-90639.95,10,6
5,2026-09-01,9149286.23,18953189.53,-9803903.30,11,7


## Excel output — `validated_output.xlsx`
5 sheets: full variance detail, exceptions log, top 5 adverse, top 5 favorable, monthly summary.
This is the file a manager actually opens and filters/pivots further.

In [17]:
with pd.ExcelWriter("validated_output.xlsx", engine="openpyxl") as writer:
    variance_final.to_excel(writer, sheet_name="Variance_Detail", index=False)
    exceptions_df.to_excel(writer, sheet_name="Exceptions_Log", index=False)
    top5_adverse.to_excel(writer, sheet_name="Top5_Adverse", index=False)
    top5_favorable.to_excel(writer, sheet_name="Top5_Favorable", index=False)
    monthly_summary.to_excel(writer, sheet_name="Monthly_Summary", index=False)

print("validated_output.xlsx written —", variance_final.shape[0], "variance rows,", len(exceptions_df), "exceptions")

validated_output.xlsx written — 103 variance rows, 13 exceptions


## Management commentary — generated from the data
Every number below is pulled from `variance_final` / `exceptions_df` / `monthly_summary` at run time.
Re-run Stage 2–4 on a new month's file and this text updates itself — that's the "automation" part, not just the Excel output.

In [18]:
latest_month = variance_final["month"].max()
total_actual_latest = variance_final.loc[variance_final["month"]==latest_month, "actual_amount"].sum()
total_budget_latest = variance_final.loc[variance_final["month"]==latest_month, "budget_amount"].sum()
total_var_latest = total_actual_latest - total_budget_latest
total_var_pct = total_var_latest / total_budget_latest

top_adverse_driver = top5_adverse.iloc[0]
top_favorable_driver = top5_favorable.iloc[0]

n_exceptions = len(exceptions_df)
n_total_rows = len(actuals)
exception_rate = n_exceptions / n_total_rows

summary_text = {
    "period": latest_month,
    "total_actual": total_actual_latest,
    "total_budget": total_budget_latest,
    "total_variance": total_var_latest,
    "total_variance_pct": total_var_pct,
    "top_adverse": f"{top_adverse_driver['cost_center']} / {top_adverse_driver['account_category']} "
                   f"({top_adverse_driver['month']}): variance {top_adverse_driver['abs_variance']:,.0f}",
    "top_favorable": f"{top_favorable_driver['cost_center']} / {top_favorable_driver['account_category']} "
                      f"({top_favorable_driver['month']}): variance {top_favorable_driver['abs_variance']:,.0f}",
    "n_exceptions": n_exceptions,
    "n_total_rows": n_total_rows,
    "exception_rate": exception_rate,
}
summary_text

{'period': '2026-09-01',
 'total_actual': np.float64(9149286.229999999),
 'total_budget': np.float64(18953189.53),
 'total_variance': np.float64(-9803903.300000003),
 'total_variance_pct': np.float64(-0.517269311557399),
 'top_adverse': 'CC-100 / Revenue (2026-09-01): variance -10,017,728',
 'top_favorable': 'CC-100 / Revenue (2026-04-01): variance 550,214',
 'n_exceptions': 13,
 'n_total_rows': 110,
 'exception_rate': 0.11818181818181818}

## Build the PDF (reportlab)

In [19]:
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import mm

styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="Small", fontSize=9, textColor=colors.grey))

doc = SimpleDocTemplate("management_summary.pdf", pagesize=A4,
                         topMargin=20*mm, bottomMargin=20*mm, leftMargin=20*mm, rightMargin=20*mm)
story = []

story.append(Paragraph("Monthly Variance Report — Management Summary", styles["Title"]))
story.append(Paragraph(f"Period: {summary_text['period']}  |  Auto-generated from validated actuals", styles["Small"]))
story.append(Spacer(1, 12))

direction = "favorable" if summary_text["total_variance"] < 0 else "unfavorable"
# note: this overall figure mixes revenue + cost lines, so sign alone isn't the full story —
# flagged explicitly below rather than asserted as simply good/bad
story.append(Paragraph(
    f"Total actuals for {summary_text['period']} were "
    f"Rs. {summary_text['total_actual']:,.0f} against a budget of Rs. {summary_text['total_budget']:,.0f}, "
    f"a variance of Rs. {summary_text['total_variance']:,.0f} "
    f"({summary_text['total_variance_pct']*100:.1f}%). "
    f"This total nets revenue and cost lines together — see Variance_Detail for the line-item picture before drawing a conclusion.",
    styles["Normal"]))
story.append(Spacer(1, 8))

story.append(Paragraph("Key drivers", styles["Heading2"]))
story.append(Paragraph(f"<b>Largest adverse driver:</b> {summary_text['top_adverse']}", styles["Normal"]))
story.append(Paragraph(f"<b>Largest favorable driver:</b> {summary_text['top_favorable']}", styles["Normal"]))
story.append(Spacer(1, 8))

story.append(Paragraph("Data quality", styles["Heading2"]))
story.append(Paragraph(
    f"{summary_text['n_exceptions']} of {summary_text['n_total_rows']} input rows "
    f"({summary_text['exception_rate']*100:.1f}%) were flagged during validation and excluded or held for review "
    f"before this analysis was run. See Exceptions_Log for the full list and reasons.",
    styles["Normal"]))
story.append(Spacer(1, 12))

story.append(Paragraph("Monthly trend", styles["Heading2"]))
table_data = [["Month","Total Actual","Total Budget","Variance","Unfav. lines","Fav. lines"]]
for _, r in monthly_summary.iterrows():
    table_data.append([r["month"], f"{r['total_actual']:,.0f}", f"{r['total_budget']:,.0f}",
                        f"{r['total_variance']:,.0f}", int(r["unfavorable_count"]), int(r["favorable_count"])])

t = Table(table_data, hAlign="LEFT")
t.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), colors.HexColor("#2c3e50")),
    ("TEXTCOLOR", (0,0), (-1,0), colors.white),
    ("FONTSIZE", (0,0), (-1,-1), 8),
    ("GRID", (0,0), (-1,-1), 0.5, colors.grey),
    ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.white, colors.HexColor("#f2f2f2")]),
]))
story.append(t)
story.append(Spacer(1, 12))

story.append(Paragraph(
    "This summary was generated by an automated validation + variance pipeline (Pandas). "
    "Exceptions require human review before final sign-off.", styles["Small"]))

doc.build(story)
print("management_summary.pdf written")

management_summary.pdf written
